# GMM vs Otsu Threshold — Real AdaND Experiments
Self-contained: clone, patch, write GMM code, run all 6 pairs.

In [ ]:
#@title **1. Clone, Install, Patch**
import os, sys, shutil, subprocess, re, time

REPO = '/content/ZS-NTTA'
DATA = '/content/datasets'

if not os.path.exists(REPO):
    !git clone https://github.com/tmlr-group/ZS-NTTA.git {REPO}
else:
    print('Repo exists.')

os.chdir(REPO)
!pip install -q ml-collections absl-py ftfy wandb seaborn scikit-learn regex scipy

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')

# Verify scipy
r = subprocess.run(['python', '-c', 'from scipy.stats import norm; print("scipy OK")'],
                   capture_output=True, text=True, cwd=REPO)
print(r.stdout.strip(), r.stderr.strip() if r.stderr.strip() else '')

In [ ]:
#@title **2. Patch paths + dataset_class_names**
os.makedirs(DATA, exist_ok=True)

# Fix hardcoded path
cfg_path = os.path.join(REPO, 'configs/default_configs.py')
with open(cfg_path, 'r') as f:
    txt = f.read()
if '/data2/chentao' in txt:
    txt = txt.replace('/data2/chentao/data/check_data', DATA)
    with open(cfg_path, 'w') as f:
        f.write(txt)
    print('Patched default_configs.py')
else:
    print('Already patched.')

# Safe dataset_class_names
DCN = os.path.join(REPO, 'data/dataset_class_names.py')
DCN_BAK = os.path.join(REPO, 'data/dataset_class_names_ORIGINAL.py')
if not os.path.exists(DCN_BAK):
    shutil.copy2(DCN, DCN_BAK)

with open(DCN, 'w') as f:
    f.write(f'import os, warnings\nfrom .imagenet_prompts import imagenet_classes, cifar10_classes, cifar100_classes\nfrom .fewshot_datasets import fewshot_datasets\nfrom .imagenet_variants import thousand_k_to_200, imagenet_a_mask, imagenet_r_mask, imagenet_v_mask\nroot = "{DATA}"\nbird200_classes = []\ncar196_classes = []\nfood101_classes = ["Apple pie","Baby back ribs","Baklava","Beef carpaccio","Beef tartare","Beet salad","Beignets","Bibimbap","Bread pudding","Breakfast burrito","Bruschetta","Caesar salad","Cannoli","Caprese salad","Carrot cake","Ceviche","Cheesecake","Cheese plate","Chicken curry","Chicken quesadilla","Chicken wings","Chocolate cake","Chocolate mousse","Churros","Clam chowder","Club sandwich","Crab cakes","Creme brulee","Croque madame","Cup cakes","Deviled eggs","Donuts","Dumplings","Edamame","Eggs benedict","Escargots","Falafel","Filet mignon","Fish and chips","Foie gras","French fries","French onion soup","French toast","Fried calamari","Fried rice","Frozen yogurt","Garlic bread","Gnocchi","Greek salad","Grilled cheese sandwich","Grilled salmon","Guacamole","Gyoza","Hamburger","Hot and sour soup","Hot dog","Huevos rancheros","Hummus","Ice cream","Lasagna","Lobster bisque","Lobster roll sandwich","Macaroni and cheese","Macarons","Miso soup","Mussels","Nachos","Omelette","Onion rings","Oysters","Pad thai","Paella","Pancakes","Panna cotta","Peking duck","Pho","Pizza","Pork chop","Poutine","Prime rib","Pulled pork sandwich","Ramen","Ravioli","Red velvet cake","Risotto","Samosa","Sashimi","Scallops","Seaweed salad","Shrimp and grits","Spaghetti bolognese","Spaghetti carbonara","Spring rolls","Steak","Strawberry shortcake","Sushi","Tacos","Takoyaki","Tiramisu","Tuna tartare","Waffles"]\npet37_classes = []\ndef get_classnames(set_id):\n    m = {{"CIFAR-100": cifar100_classes, "CIFAR-10": cifar10_classes, "CIFAR-100-C": cifar100_classes, "CIFAR-10-C": cifar10_classes}}\n    if set_id in m: return m[set_id]\n    if set_id in ["A","R","K","V","I","ImageNet-C"]: return imagenet_classes\n    raise ValueError(set_id)\n')
print('Rewrote dataset_class_names.py')

for d in ['data/noisy_data_idx/0.33', 'data/noisy_data_idx/1.0', 'data/noisy_data_idx/3.0',
          'results/main_results', 'data_analysis/score', 'data_analysis/conf_pkl']:
    os.makedirs(os.path.join(REPO, d), exist_ok=True)
print('Dirs ready.')

In [ ]:
#@title **3. Patch OODDetector + write enhanced_classifier**

# Enhanced detector
with open(os.path.join(REPO, 'clip/enhanced_classifier.py'), 'w') as f:
    f.write('import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nclass EnhancedOODDetector(nn.Module):\n    def __init__(self, input_size=512, hidden_size=128, use_augmented_features=True):\n        super().__init__()\n        self.use_augmented_features = use_augmented_features\n        dim = input_size + 4 if use_augmented_features else input_size\n        self.net = nn.Sequential(nn.Linear(dim, hidden_size), nn.ReLU(inplace=True), nn.Linear(hidden_size, 2))\n\n    def forward(self, x, clip_output=None):\n        if self.use_augmented_features and clip_output is not None:\n            aug = self._aug(x, clip_output)\n            x = torch.cat([x, aug], dim=-1)\n        return self.net(x)\n\n    def _aug(self, feats, clip_out):\n        norm = torch.norm(feats, dim=-1, keepdim=True)\n        p = F.softmax(clip_out, dim=-1)\n        mcm = p.max(dim=-1, keepdim=True)[0]\n        ent = -(p * F.log_softmax(clip_out, dim=-1)).sum(dim=-1, keepdim=True)\n        t2 = p.topk(min(2, p.shape[-1]), dim=-1)[0]\n        gap = (t2[:, 0] - t2[:, 1]).unsqueeze(-1) if t2.shape[-1] >= 2 else t2[:, 0:1]\n        return torch.cat([norm, mcm, ent, gap], dim=-1)\n')
print('Wrote enhanced_classifier.py')

# Patch OODDetector only
clf_path = os.path.join(REPO, 'clip/classifier.py')
with open(clf_path, 'r') as f:
    clf_text = f.read()
old_block = 'class OODDetector(nn.Module):\n    def __init__(self, input_size=512, hidden_size=256):\n        super(OODDetector, self).__init__()\n        self.fc = nn.Linear(input_size, 2)\n\n    def forward(self, x):\n        x = self.fc(x)\n        return x'
new_block = 'class OODDetector(nn.Module):\n    def __init__(self, input_size=512, hidden_size=256):\n        super(OODDetector, self).__init__()\n        self.fc = nn.Linear(input_size, 2)\n\n    def forward(self, x, clip_output=None):\n        x = self.fc(x)\n        return x'
if 'def forward(self, x, clip_output=None)' not in clf_text:
    clf_text = clf_text.replace(old_block, new_block)
    with open(clf_path, 'w') as f:
        f.write(clf_text)
    print('Patched OODDetector')
else:
    print('Already patched.')

In [ ]:
#@title **4. Write zs_noisytta.py with GMM support (scipy imported locally)**

orig = os.path.join(REPO, 'ttda_method/zs_noisytta.py')
bak = os.path.join(REPO, 'ttda_method/zs_noisytta_ORIGINAL.py')
if not os.path.exists(bak):
    shutil.copy2(orig, bak)

with open(orig, 'w') as f:
    f.write('''import torch
import torch.nn.parallel
import torch.nn.functional as F
import torch.nn as nn
import torch.optim
import torch.utils.data
import torch.utils.data.distributed
import numpy as np
from collections import deque
import copy

from .zs_clip import ZeroShotCLIP
from clip.classifier import OODDetector
from clip.enhanced_classifier import EnhancedOODDetector
from utils.utils import *


class ZeroShotNTTA(ZeroShotCLIP):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        fd = {"ViT-L/14": 768, "ViT-B/16": 512}.get(self.args.model.arch, 1024)
        dev = self.model.image_encoder.conv1.weight.device

        enh = getattr(self.args.inference, "use_enhanced_detector", False)
        hs = getattr(self.args.inference, "detector_hidden_size", 128)
        if enh:
            self.ood_net = EnhancedOODDetector(fd, hs, True).to(dev)
        else:
            self.ood_net = OODDetector(fd).to(dev)

        # Threshold method
        self.threshold_method = getattr(self.args.inference, "threshold_method", "otsu")

        # Training flags
        self.use_soft = getattr(self.args.inference, "use_soft_labels", False)
        self.soft_alpha = getattr(self.args.inference, "soft_label_alpha", 10.0)
        self.asymmetric = getattr(self.args.inference, "asymmetric_soft", False)
        self.w_id = getattr(self.args.inference, "w_id", 1.0)
        self.w_ood = getattr(self.args.inference, "w_ood", 1.0)
        self.training_margin = getattr(self.args.inference, "training_margin", 0.0)
        self.use_smooth = getattr(self.args.inference, "use_smooth_transition", False)
        self.ramp = getattr(self.args.inference, "transition_ramp_steps", 5)
        self.ensemble_weight = getattr(self.args.inference, "ensemble_weight", 0.0)
        self.deferral_margin = getattr(self.args.inference, "deferral_margin", 0.0)

        flags = [f"threshold={self.threshold_method}"]
        if enh: flags.append("enh_det")
        if self.use_soft: flags.append(f"soft(a={self.soft_alpha})")
        if self.asymmetric: flags.append("asym")
        if self.w_id != 1.0: flags.append(f"w_id={self.w_id}")
        if self.training_margin > 0: flags.append(f"3way={self.training_margin}")
        if self.use_smooth: flags.append("smooth")
        if self.ensemble_weight > 0: flags.append(f"ens={self.ensemble_weight}")
        if self.deferral_margin > 0: flags.append(f"defer={self.deferral_margin}")
        print(f"[config] {\', \'.join(flags)}")

        self.kl_loss = nn.KLDivLoss(reduction="batchmean")
        self.ce_loss_none = nn.CrossEntropyLoss(reduction="none")
        self.ce_loss = nn.CrossEntropyLoss()
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = get_optimizer(self.args, self.ood_net.parameters(), lr=self.args.optim.lr)
        self.os_detector_queue = []
        self.loss_history = []
        ql = self.args.inference.ttda_queue_length
        self.queues = {
            "ood_detector_out_queue": deque(maxlen=ql),
            "unseen_mask_queue": deque(maxlen=ql),
            "target_queue": deque(maxlen=ql),
            "clip_output_queue": deque(maxlen=ql),
        }
        self.ttda_queue = []

    def _get_otsu_threshold(self, queue):
        if len(queue) < 10:
            return 0.5
        arr = np.array(queue)
        tr = np.arange(0, 1, 0.01)
        cs = [compute_os_variance(arr, t) for t in tr]
        return tr[np.argmin(cs)]

    def _get_gmm_threshold(self, queue, n_iter=50):
        from scipy.stats import norm as scipy_norm
        if len(queue) < 10:
            return 0.5
        s = np.sort(np.array(queue))
        mid = np.median(s)
        mu0 = s[s < mid].mean() if (s < mid).any() else mid - 0.1
        mu1 = s[s >= mid].mean() if (s >= mid).any() else mid + 0.1
        sig0 = max(s.std() * 0.5, 0.01)
        sig1 = max(s.std() * 0.5, 0.01)
        pi0 = 0.5
        for _ in range(n_iter):
            p0 = pi0 * scipy_norm.pdf(s, mu0, sig0 + 1e-8)
            p1 = (1 - pi0) * scipy_norm.pdf(s, mu1, sig1 + 1e-8)
            total = p0 + p1 + 1e-10
            gamma = p1 / total
            n0 = (1 - gamma).sum() + 1e-8
            n1 = gamma.sum() + 1e-8
            mu0 = ((1 - gamma) * s).sum() / n0
            mu1 = (gamma * s).sum() / n1
            sig0 = max(np.sqrt(((1 - gamma) * (s - mu0)**2).sum() / n0), 0.01)
            sig1 = max(np.sqrt((gamma * (s - mu1)**2).sum() / n1), 0.01)
            pi0 = n0 / len(s)
        x = np.linspace(0, 1, 1000)
        post0 = pi0 * scipy_norm.pdf(x, mu0, sig0)
        post1 = (1 - pi0) * scipy_norm.pdf(x, mu1, sig1)
        cross = np.where(np.diff(np.sign(post0 - post1)))[0]
        if len(cross) > 0:
            thr = x[cross[0]]
        else:
            thr = (mu0 + mu1) / 2
        print(f"[GMM] mu0={mu0:.3f} sig0={sig0:.3f} mu1={mu1:.3f} sig1={sig1:.3f} pi0={pi0:.3f} thr={thr:.3f}")
        return thr

    def _get_threshold(self, queue):
        if self.threshold_method == "gmm":
            return self._get_gmm_threshold(queue)
        else:
            return self._get_otsu_threshold(queue)

    def get_unseen_mask(self, clip_output, image, image_feature_raw, step, target):
        unseen_mask = super().get_unseen_mask(clip_output, image, image_feature_raw, step, target)
        self.model.eval()
        self.ood_net.train()

        with torch.enable_grad():
            if self.args.inference.batch_size == 1:
                ood_out = self.ood_net(image_feature_raw, clip_output=clip_output)
                self.ttda_queue.extend(ood_out)
                self.queues["ood_detector_out_queue"].append(ood_out)
                self.queues["unseen_mask_queue"].append(unseen_mask)
                self.queues["target_queue"].append(target)
                self.queues["clip_output_queue"].append(clip_output)
                if step != 0 and step % self.args.inference.ttda_queue_length == 0:
                    bdo = torch.stack(list(self.queues["ood_detector_out_queue"]), 0).squeeze(1)
                    bum = torch.stack(list(self.queues["unseen_mask_queue"]), 0).squeeze(1)
                    btg = torch.stack(list(self.queues["target_queue"]), 0).squeeze(1)
                    bco = torch.stack(list(self.queues["clip_output_queue"]), 0).squeeze(1)
                    self.update_detector(bdo, bum, btg, step, bco)
            else:
                ood_out = self.ood_net(image_feature_raw, clip_output=clip_output)
                self.update_detector(ood_out, unseen_mask, target, step, clip_output)

        bs = self.args.inference.batch_size
        ql = self.args.inference.ttda_queue_length if bs == 1 else bs
        start = self.args.inference.using_ttda_step * ql
        cur = step * bs

        if cur > start:
            pred = F.softmax(ood_out, 1)
            det_score = pred[:, 0]
            self.os_detector_queue.extend(det_score.detach().cpu().tolist())
            self.os_detector_queue = self.os_detector_queue[-self.args.inference.queue_length:]

            if self.args.inference.threshold_type == "adaptive":
                thr = self._get_threshold(self.os_detector_queue)
            else:
                thr = self.args.inference.fixed_threshold
            print(thr)

            unseen_mask = (det_score > thr)

            if self.use_smooth:
                a = min(1.0, (cur - start) / max(ql * self.ramp, 1))
                if a < 1.0:
                    cp = F.softmax(clip_output, 1)
                    cs = 1 - cp.max(1)[0]
                    unseen_mask = (a * det_score + (1 - a) * cs > thr)

            if self.ensemble_weight > 0:
                w = self.ensemble_weight
                cp = F.softmax(clip_output, 1)
                unseen_mask = ((1 - w) * det_score + w * (1 - cp.max(1)[0]) > thr)

            if self.deferral_margin > 0:
                unc = torch.abs(pred[:, 0] - 0.5) < self.deferral_margin
                if unc.any():
                    cp = F.softmax(clip_output, 1)
                    ct = self._get_threshold(self.os_inference_queue)
                    unseen_mask[unc] = ((1 - cp.max(1)[0]) > ct)[unc]

            return unseen_mask, pred[:, 1]

        return unseen_mask

    def update_detector(self, ood_out, unseen_mask, target, step, clip_output=None):
        logit = F.softmax(clip_output, dim=1)
        conf, _ = logit.max(1)
        unseen_mask[target == -1000] = True

        if self.training_margin > 0:
            mcm = logit.max(1)[0]
            mcm_thr = 1.0 - self._get_threshold(self.os_inference_queue) if len(self.os_inference_queue) > 10 else 0.5
            confident = torch.abs(mcm - mcm_thr) > self.training_margin
            confident[target == -1000] = True
            si = torch.where(~unseen_mask & confident)[0]
            so = torch.where(unseen_mask & confident)[0]
        else:
            si = torch.where(~unseen_mask)[0]
            so = torch.where(unseen_mask)[0]

        sa = torch.cat([si, so])
        if len(sa) < 4:
            return

        if self.use_soft and clip_output is not None:
            loss = self._soft_loss(ood_out, target, clip_output, si, so, sa)
        elif self.w_id != 1.0 or self.w_ood != 1.0:
            lab = torch.cat([torch.ones(len(si)), torch.zeros(len(so))]).cuda()
            ps = self.ce_loss_none(ood_out[sa], lab.long())
            wt = torch.ones_like(ps)
            wt[:len(si)] = self.w_id
            wt[len(si):] = self.w_ood
            loss = (ps * wt).mean()
        else:
            lab = torch.cat([torch.ones(len(si)), torch.zeros(len(so))]).cuda()
            loss = self.ce_loss(ood_out[sa], lab.long())

        self.optimizer.zero_grad()
        loss.backward()
        self.loss_history.append(loss.item())
        self.optimizer.step()

    def _soft_loss(self, ood_out, target, clip_output, si, so, sa):
        a = self.soft_alpha
        p = F.softmax(clip_output, 1)
        mcm = p.max(1)[0]
        mcm_thr = 1.0 - self._get_threshold(self.os_inference_queue)

        soft = torch.zeros(len(sa), 2, device=ood_out.device)
        if len(si) > 0:
            c = torch.sigmoid(a * (mcm[si] - mcm_thr))
            soft[:len(si), 1] = c
            soft[:len(si), 0] = 1 - c
        if len(so) > 0:
            off = len(si)
            if self.asymmetric:
                soft[off:, 0] = 1.0
                soft[off:, 1] = 0.0
            else:
                c = torch.sigmoid(a * (mcm_thr - mcm[so]))
                c[target[so] == -1000] = 1.0
                soft[off:, 0] = c
                soft[off:, 1] = 1 - c

        soft = soft / soft.sum(1, keepdim=True).clamp(min=1e-8)
        lp = F.log_softmax(ood_out[sa], 1)
        return self.kl_loss(lp, soft.detach())
''')

print('Wrote zs_noisytta.py with GMM support (scipy imported locally).')

# Verify the file is correct
with open(orig, 'r') as f:
    content = f.read()
assert 'from scipy.stats import norm' not in content, 'ERROR: scipy still at top level!'
assert 'scipy_norm' in content, 'ERROR: local scipy import missing!'
assert '_get_gmm_threshold' in content, 'ERROR: GMM method missing!'
assert 'threshold_method' in content, 'ERROR: threshold_method flag missing!'
print('Verification passed: no top-level scipy, GMM method present.')

In [ ]:
#@title **5. Setup OOD datasets**
os.makedirs(os.path.join(DATA, '_ood_raw'), exist_ok=True)

try:
    !pip install -q pytorch-ood
    import pytorch_ood.dataset.img as ood_data
    ds = ood_data.LSUNCrop(root=os.path.join(DATA, '_ood_raw'), download=True)
    !mkdir -p {DATA}/LSUN/0
    !ln -sf {DATA}/_ood_raw/LSUN/test/*.png {DATA}/LSUN/0/ 2>/dev/null
    !ln -sf {DATA}/_ood_raw/LSUN/test/*.jpg {DATA}/LSUN/0/ 2>/dev/null
    print(f'LSUN ready')
except Exception as e:
    print(f'LSUN: {e}')

try:
    ds = ood_data.Places365(root=os.path.join(DATA, '_ood_raw'), download=True)
    !mkdir -p {DATA}/ImageNet_OOD_dataset
    !ln -sf {DATA}/_ood_raw/places365 {DATA}/ImageNet_OOD_dataset/Places
    print('Places ready.')
except Exception as e:
    print(f'Places: {e}')

In [ ]:
#@title **6. Create configs**

def cfg_thr(name, method='otsu'):
    with open(os.path.join(REPO, f'configs/{name}'), 'w') as f:
        f.write(f'''from configs.default_configs import get_default_configs
def get_config():
    c = get_default_configs()
    c.method = "ZS-NTTA"
    c.training.save_ckpt = True
    c.training.gaussian_sampling = False
    i = c.inference
    i.ttda_queue_length = 64
    i.top = 1.
    i.update_classifier = "None"
    i.inject_noise_type = "gaussian"
    i.using_ttda_step = 10
    i.gaussian_rate = 0.125
    i.batch_size = 128
    i.threshold_method = "{method}"
    i.use_enhanced_detector = False
    i.detector_hidden_size = 128
    i.use_soft_labels = False
    i.soft_label_alpha = 10.0
    i.asymmetric_soft = False
    i.use_smooth_transition = False
    i.transition_ramp_steps = 5
    i.w_id = 1.0
    i.w_ood = 1.0
    i.training_margin = 0.0
    i.ensemble_weight = 0.0
    i.deferral_margin = 0.0
    o = c.optim
    o.classifier_lr = 0.0005
    o.lr = 0.0005
    o.weight_decay = 0
    o.optimizer = "Adam"
    o.beta1 = 0.9
    return c
''')
    print(f'  Created {name}')

cfg_thr('thr_otsu.py', method='otsu')
cfg_thr('thr_gmm.py',  method='gmm')

In [ ]:
#@title **7. Experiment runner**

def run(config, id_set, ood_set, gpu=0, label=None):
    label = label or config.replace('.py', '')
    print(f'\n{"="*55}\n  {label}  |  ID={id_set}  OOD={ood_set}\n{"="*55}')
    # Clear cached results
    rd = os.path.join(REPO, f'results/main_results/{id_set}')
    if os.path.isdir(rd):
        for sub in os.listdir(rd):
            sd = os.path.join(rd, sub)
            if os.path.isdir(sd):
                for ff in os.listdir(sd):
                    if ood_set in ff:
                        os.remove(os.path.join(sd, ff))
    cmd = ['python', 'main.py', f'--config=configs/{config}',
           f'--test_set={id_set}', f'--OOD_set={ood_set}', f'--gpu={gpu}']
    t0 = time.time()
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=3600, cwd=REPO)
        out = r.stdout + '\n' + r.stderr
    except Exception as e:
        print(f'  ERROR: {e}')
        return {'label': label, 'id': id_set, 'ood': ood_set, 'error': str(e)}
    dt = time.time() - t0
    m = {}
    for line in out.split('\n'):
        if 'ACC_S:' in line and 'ACC_N:' in line and 'ACC_H:' in line:
            ms = re.search(r'ACC_S:\s*([\d.]+)', line)
            mn = re.search(r'ACC_N:\s*([\d.]+)', line)
            mh = re.search(r'ACC_H:\s*([\d.]+)', line)
            if ms and mn and mh:
                m = {'AccS': float(ms.group(1)), 'AccN': float(mn.group(1)), 'AccH': float(mh.group(1))}
    if m:
        print(f'  AccS={m["AccS"]:.2f}  AccN={m["AccN"]:.2f}  AccH={m["AccH"]:.2f}  ({dt:.0f}s)')
    else:
        print('  Parse failed. Last lines:')
        for l in out.strip().split('\n')[-15:]:
            if l.strip(): print(f'    {l.strip()}')
    return {'label': label, 'id': id_set, 'ood': ood_set, **m}

print('Runner ready.')

In [ ]:
#@title **8. Quick sanity check — verify GMM is actually running**

# First check: does the config flag load?
print('=== Config check ===')
r = subprocess.run(
    ['python', '-c', '''import sys; sys.path.insert(0,"."); from configs.thr_gmm import get_config; c=get_config(); print(f"threshold_method={c.inference.threshold_method}")'''],
    capture_output=True, text=True, cwd=REPO)
print(r.stdout.strip())
if 'gmm' in r.stdout:
    print('Config OK: GMM flag loads correctly.')
else:
    print(f'CONFIG ERROR! stderr: {r.stderr[:200]}')

# Second check: does GMM code execute?
print('\n=== GMM execution check ===')
r = subprocess.run(
    ['python', 'main.py', '--config=configs/thr_gmm.py',
     '--test_set=CIFAR-10', '--OOD_set=SVHN', '--gpu=0'],
    capture_output=True, text=True, timeout=300, cwd=REPO)
out = r.stdout + '\n' + r.stderr

# Check for [config] and [GMM] lines
config_lines = [l for l in out.split('\n') if '[config]' in l]
gmm_lines = [l for l in out.split('\n') if '[GMM]' in l]

if config_lines:
    print(f'Config loaded: {config_lines[0].strip()}')
else:
    print('WARNING: No [config] line found!')

if gmm_lines:
    print(f'GMM executing: {len(gmm_lines)} threshold computations')
    print(f'  First: {gmm_lines[0].strip()}')
    print(f'  Last:  {gmm_lines[-1].strip()}')
else:
    print('ERROR: No [GMM] lines found! GMM is NOT running.')
    print('Last 10 lines of output:')
    for l in out.strip().split('\n')[-10:]:
        print(f'  {l.strip()}')

In [ ]:
#@title **9. Run ZS-CLIP baselines**
for id_set in ['CIFAR-10', 'CIFAR-100']:
    for ood in ['SVHN', 'LSUN', 'Places']:
        run('zs_clip_configs.py', id_set, ood, label='ZS-CLIP')

In [ ]:
#@title **10. Run Otsu vs GMM on all 6 pairs**

R = []
for id_set in ['CIFAR-10', 'CIFAR-100']:
    for ood in ['SVHN', 'LSUN', 'Places']:
        R.append(run('thr_otsu.py', id_set, ood, label='Otsu'))
        R.append(run('thr_gmm.py',  id_set, ood, label='GMM'))

# Print results
pairs = [('CIFAR-10','SVHN'), ('CIFAR-10','LSUN'), ('CIFAR-10','Places'),
         ('CIFAR-100','SVHN'), ('CIFAR-100','LSUN'), ('CIFAR-100','Places')]

for metric in ['AccH', 'AccS', 'AccN']:
    print(f'\n{"="*85}')
    print(f'  {metric}')
    print(f'{"="*85}')
    print(f'{"Method":<8s}', end='')
    for i, o in pairs:
        print(f' {i[:4]}+{o[:5]:>5s}', end='')
    print(f'   {"Avg":>5s}')
    print(f'{"-"*85}')
    for lbl in ['Otsu', 'GMM']:
        row = f'{lbl:<8s}'
        vals = []
        for i, o in pairs:
            m = [r for r in R if r['label']==lbl and r.get('id')==i and r.get('ood')==o and metric in r]
            if m:
                v = m[0][metric]
                vals.append(v)
                row += f' {v:>10.2f}'
            else:
                row += f' {"---":>10s}'
        if vals:
            row += f'   {sum(vals)/len(vals):>5.2f}'
        print(row)